In [1]:

# IMPORTS
import numpy as np
import pandas as pd
import requests


#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt
     

Question 1 What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?

In [7]:
from io import StringIO

In [8]:
def get_ipos_data() -> pd.DataFrame:
    
    url = f"https://www.iposcoop.com/ipos-recently-filed/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found.")

        return tables[0]

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [9]:
ipos = get_ipos_data()

In [10]:
ipos.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
4,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Week of,S/O


In [16]:
ipos_withdrawl = ipos[
    (ipos['Expected To Trade'] == 'Withdrawn') 
    & (ipos['File Date'] <= '2026-09-11')
    ]
ipos_withdrawl.head()
len(ipos_withdrawl)

31

In [17]:
rules = {
    "Technologies": [r"\bTechnologies\b"],
    "Acquisition Corp": [
        r"\bAcquisition Corp\b",
        r"\bAcquisition Corporation\b",
        r"\bCorp\b"
    ],
    "Inc.": [
        r"\bInc\b",
        r"\bIncorporated\b"
    ],
    "Group": [r"\bGroup\b"],
    "Limited": [
        r"\bLtd\b",
        r"\bLimited\b"
    ],
    "Holdings": [
        r"\bHoldings\b",
        r"\bHolding\b"
    ]
}


In [18]:
ipos_withdrawl["Company Type"] = "Other"

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/2988296722.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Company Type"] = "Other"


In [21]:
for company_type, patterns in rules.items():
    pattern = "|".join(patterns)

    mask = (
        ipos_withdrawl["Company Type"].eq("Other") &
        ipos_withdrawl["Company"].str.contains(pattern, regex=True, na=False)
    )

    ipos_withdrawl.loc[mask, "Company Type"] = company_type

In [22]:
ipos_withdrawl["Company Type"].value_counts()

Company Type
Other               8
Limited             7
Acquisition Corp    5
Holdings            4
Technologies        3
Group               3
Inc.                1
Name: count, dtype: int64

In [23]:
def extract_numeric(value):
    return float(
        str(value)
        .replace("$", "")
        .replace(",", "")
        .strip()
    )

In [24]:
price_low = ipos_withdrawl["Price Low"].apply(extract_numeric)
price_high = ipos_withdrawl["Price High"].apply(extract_numeric)

ipos_withdrawl["Avg_price"] = (price_low + price_high) / 2

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/2081960412.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Avg_price"] = (price_low + price_high) / 2


In [25]:
ipos_withdrawl["Shares (millions)"] = pd.to_numeric(
    ipos_withdrawl["Shares (millions)"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False),
    errors="coerce"
)

ipos_withdrawl["Est $ Vol (millions)"] = pd.to_numeric(
    ipos_withdrawl["Est $ Vol (millions)"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
    errors="coerce"
)

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/764667097.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Shares (millions)"] = pd.to_numeric(
/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/764667097.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Est $ Vol (millions)"] = pd.to_numeric(


In [27]:
ipos_withdrawl["Shares_offered_value"] = (
    ipos_withdrawl["Shares (millions)"] * ipos_withdrawl["Avg_price"]
).fillna(ipos_withdrawl["Est $ Vol (millions)"])

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/2522762365.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Shares_offered_value"] = (


In [28]:
ipos_withdrawl.groupby("Company Type")["Shares_offered_value"].sum()

Company Type
Acquisition Corp    499.9850
Group                32.5000
Holdings            311.6575
Inc.                351.0000
Limited             197.2500
Other               290.4450
Technologies        184.9000
Name: Shares_offered_value, dtype: float64